In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np

import ast
from sklearn.metrics import mean_absolute_percentage_error 
from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import holidays
import lightgbm as lgb

In [3]:
df=pd.read_csv("/kaggle/input/crop-data/new.csv")

In [4]:
df.head()

,location,crop,price_trend,date,raw_prices
0,"Manvi, Raichur, Karnataka",Jowar(Sorghum)/per kg,₹9 → ₹24/kg,2019-06-21,"{""datetime(2019, 6, 12)"": 9.0, ""datetime(2019,..."
1,"Amroha, Jyotiba Phule Nagar, Uttar Pradesh",Arhar+Dal(Tur+Dal)/per kg,₹78 → ₹88/kg,2019-11-18,"{""datetime(2019, 11, 9)"": 77.6, ""datetime(2019..."
2,"Bewar, Mainpuri, Uttar Pradesh",Cucumbar(Kheera)/per kg,₹6 → ₹8/kg,2019-05-24,"{""datetime(2019, 5, 15)"": 6.0, ""datetime(2019,..."
3,"Kharupetia, Darrang, Assam",Cluster+beans/per kg,₹35 → ₹37/kg,2019-10-03,"{""datetime(2019, 9, 24)"": 35.0, ""datetime(2019..."
4,"Lakhani, Banaskanth, Gujarat",Rajgir/per kg,₹59 → ₹59/kg,2019-08-07,"{""datetime(2019, 7, 29)"": 58.85, ""datetime(201..."


In [5]:
import pandas as pd
import numpy as np
import ast
import pickle

from datetime import datetime, timedelta
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_absolute_error,
    r2_score
)

import lightgbm as lgb

In [21]:


def process_raw_data(raw_df):
    processed = []
    for idx, row in raw_df.iterrows():
        try:
            cleaned = row["raw_prices"].replace("datetime", "") \
                                       .replace("(", "") \
                                       .replace(")", "")
            tmp = ast.literal_eval(cleaned)    
            price_dict = {
                datetime.strptime(k.strip(), "%Y, %m, %d"): v
                for k, v in tmp.items()
            }
            sorted_dates = sorted(price_dict)
            for i in range(len(sorted_dates) - 9):
                d0 = sorted_dates[i]
                d9 = sorted_dates[i + 9]
                processed.append({
                    "state":        (lambda parts: parts[-1] if len(parts)>=3 else "Unknown")(
                                        [x.strip() for x in row["location"].split(",")]),
                    "district":     (lambda parts: parts[-2] if len(parts)>=2 else "Unknown")(
                                        [x.strip() for x in row["location"].split(",")]),
                    "market":       (lambda parts: parts[-3])(
                                        [x.strip() for x in row["location"].split(",")]),
                    "crop":         row["crop"],                
                    "current_date": d0,
                    "current_price":price_dict[d0],
                    "target_date":  d9,
                    "target_price": price_dict[d9],
                })
        except Exception as e:
            print(f"Error processing row {idx!r}: {e}")
            continue

    return pd.DataFrame(processed)


In [7]:
def add_features(df):
    df['day_of_year']   = df['current_date'].dt.dayofyear
    df['month']         = df['current_date'].dt.month
    df['3day_momentum'] = df.groupby(
        ['state', 'district', 'market', 'crop']
    )['current_price'].pct_change(3)

    encoder = {}
    for col in ['state', 'district', 'market', 'crop']:
        le = LabelEncoder()
        df[col + '_code'] = le.fit_transform(df[col].astype(str))
        encoder[col] = le

    final_df = df[[
        'state_code','district_code','market_code','crop_code',
        'day_of_year','month','3day_momentum',
        'current_price','target_price','target_date'
    ]].dropna()
    return final_df, encoder

In [27]:
def add_featuress(df):
    df['day_of_year']   = df['current_date'].dt.dayofyear
    df['month']         = df['current_date'].dt.month
    df['3day_momentum'] = df.groupby(
        ['state', 'district', 'market', 'crop']
    )['current_price'].pct_change(3)

    encoder = {}
    for col in ['state', 'district', 'market', 'crop']:
        le = LabelEncoder()
        df[col + '_code'] = le.fit_transform(df[col].astype(str))
        encoder[col] = le

    final_df = df[[
        'state_code','district_code','market_code','crop_code',
        'day_of_year','month','3day_momentum',
        'current_price','target_price','target_date'
    ]].dropna()
    return final_df

In [34]:
def train_price_model(df):
    features = [
        'state_code','district_code','market_code','crop_code',
        'day_of_year','month','3day_momentum','current_price'
    ]
    X_train, X_val, y_train, y_val = train_test_split(
        df[features], df['target_price'], test_size=0.2, random_state=42
    )

    train_data = lgb.Dataset(X_train, label=y_train)
    val_data   = lgb.Dataset(X_val, label=y_val, reference=train_data)

    params = {
        'objective':        'regression',
        'metric':           'mape',
        'n_estimators':200,
        'max_depth':-1,
        'num_leaves':       50,
        'learning_rate':    0.1,
        'bagging_freq':0,
        'bagging_fraction': 0.8,
        'verbosity':        -1
    }

    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
    )
    return model, X_val, y_val

In [9]:
class CropPricePredictor:
    def __init__(self, model, label_encoders):
        self.model    = model
        self.encoders = label_encoders

    def predict(self, input_data):
        current_date = datetime.strptime(
            input_data['current_date'], "%Y-%m-%d"
        )

        features = {
            'state_code':     self._encode('state',    input_data['state']),
            'district_code':  self._encode('district', input_data['district']),
            'market_code':    self._encode('market',   input_data['market']),
            'crop_code':      self._encode('crop',     input_data['crop']),
            'day_of_year':    current_date.timetuple().tm_yday,
            'month':          current_date.month,
            '3day_momentum':  0.0,
            'current_price':  input_data['current_price']
        }
        price_pred   = self.model.predict(pd.DataFrame([features]))[0]
        target_date  = current_date + timedelta(days=9)
        day          = target_date.day
        month_name   = target_date.strftime("%B")
        prediction   = round(price_pred, 2)
        text         = f"{prediction}/kg by {month_name} {day}"

        return {'predicted_price': text}

    def _encode(self, feature_type, value):
        return self.encoders[feature_type].transform([value])[0]


In [36]:

if __name__ == "__main__":
    raw_df        = pd.read_csv('/kaggle/input/crop-data/new.csv')
    processed_df  = process_raw_data(raw_df)
    final_df, encoders = add_features(processed_df)
    model, X_val, y_val = train_price_model(final_df)
    y_pred = model.predict(X_val)
    mape   = mean_absolute_percentage_error(y_val, y_pred)
    mae    = mean_absolute_error(y_val, y_pred)
    r2     = r2_score(y_val, y_pred)
    print(f'Validation MAPE = {mape:.4f}')
    print(f'Validation MAE  = {mae:.4f}')
    print(f'Validation R²   = {r2:.4f}')
    predictor = CropPricePredictor(model, encoders)
    sample_input = {
        'state':         'Kerala',
        'district':      'Kollam',
        'market':        'Anchal',
        'crop':          'Amaranthus/per kg',
        'current_date':  '2024-05-02',
        'current_price':  76.55
    }
    print("Sample prediction:", predictor.predict(sample_input)['predicted_price'])
    with open('new_model.pkl', 'wb') as f:
        pickle.dump({
            'model': model,
            'encoders': encoders
        }, f)
    print('Trained LGB model was saved!')


/usr/local/lib/python3.11/dist-packages/lightgbm/engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


Validation MAPE = 0.3568
Validation MAE  = 27.3411
Validation R²   = 0.8672
Sample prediction: 471.23/kg by May 11
Trained LGB model was saved!


In [33]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
new_df=process_raw_data(df)
dff=add_featuress(new_df)
print(dff.head())

features = [
        'state_code','district_code','market_code','crop_code',
        'day_of_year','month','3day_momentum','current_price'
    ]
X_train, X_val, y_train, y_val = train_test_split(
        dff[features], dff['target_price'], test_size=0.2, random_state=42
    )
lgb_model = lgb.LGBMRegressor()

param_grid = {
    'num_leaves': [31, 50],
    'max_depth': [-1, 10, 20],
    'learning_rate': [0.01, 0.1],
    'n_estimators': [100, 200],
    'bagging_fraction': [0.8, 1.0],
    'bagging_freq': [0, 1],
}
grid = GridSearchCV(
    estimator=lgb_model,
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=1
)
grid.fit(X_train,y_train)
print("Best parameters found: ", grid.best_params_)
print("Best CV score (negative MSE): ", grid.best_score_)


      state_code  district_code  market_code  crop_code  day_of_year  month  \
1448          26            307         1196         41          197      7   
1652          26            418         1679        213          243      8   
1957          26             44          183        169          306     11   
2268          26            355         1399         96           69      3   
2600          12            327         1255        119          344     12   

      3day_momentum  current_price  target_price target_date  
1448      -0.027027          18.00         19.80  2019-07-25  
1652       0.187500          19.00         20.00  2019-09-09  
1957      -0.125000          14.00         15.00  2019-11-11  
2268      -0.100000          33.75         34.30  2021-03-19  
2600      -0.061224          46.00         53.51  2019-12-19  
Fitting 3 folds for each of 96 candidates, totalling 288 fits
Best parameters found:  {'bagging_fraction': 0.8, 'bagging_freq': 0, 'learning_rate':

In [32]:
import sklearn
sklearn.metrics.get_scorer_names()

['accuracy',
 'adjusted_mutual_info_score',
 'adjusted_rand_score',
 'average_precision',
 'balanced_accuracy',
 'completeness_score',
 'explained_variance',
 'f1',
 'f1_macro',
 'f1_micro',
 'f1_samples',
 'f1_weighted',
 'fowlkes_mallows_score',
 'homogeneity_score',
 'jaccard',
 'jaccard_macro',
 'jaccard_micro',
 'jaccard_samples',
 'jaccard_weighted',
 'matthews_corrcoef',
 'max_error',
 'mutual_info_score',
 'neg_brier_score',
 'neg_log_loss',
 'neg_mean_absolute_error',
 'neg_mean_absolute_percentage_error',
 'neg_mean_gamma_deviance',
 'neg_mean_poisson_deviance',
 'neg_mean_squared_error',
 'neg_mean_squared_log_error',
 'neg_median_absolute_error',
 'neg_negative_likelihood_ratio',
 'neg_root_mean_squared_error',
 'normalized_mutual_info_score',
 'positive_likelihood_ratio',
 'precision',
 'precision_macro',
 'precision_micro',
 'precision_samples',
 'precision_weighted',
 'r2',
 'rand_score',
 'recall',
 'recall_macro',
 'recall_micro',
 'recall_samples',
 'recall_weighted',